# What is MCP?

**Model Context Protocol (MCP)** is an open standard that lets AI apps (Claude, Cursor, your own code) talk to tools and data in one consistent way.

Think of it like **USB-C for AI**: one plug, many devices.

| Role       | What it is                                 | Example                          |
| ---------- | ------------------------------------------ | -------------------------------- |
| **Host**   | The AI app the user uses                   | Claude Desktop, Cursor           |
| **Client** | Lives inside the host, talks to one server | built into the host              |
| **Server** | Exposes tools, resources, prompts          | the one we'll build in Episode 1 |

Servers offer three things:

- **Tools**: functions the model can call (`add`, `search_web`)
- **Resources**: data the model can read (files, notes)
- **Prompts**: reusable prompt templates


## Under the hood: it's just JSON-RPC messages

No install needed for this episode. Here are the three messages that matter most.


In [1]:
import json

# 1. Client connects to the MCP server and says hello
initialize = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": "2025-06-18",
        "capabilities": {},
        "clientInfo": {"name": "my-client", "version": "0.1"},
    },
}

# 2. Client asks the MCP server: what tools do you have?
list_tools = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/list",
}

# 3. Client calls a tool on the MCP server
call_tool = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {"name": "add", "arguments": {"a": 2, "b": 3}},
}

for msg in (initialize, list_tools, call_tool):
    print(json.dumps(msg, indent=2))
    print("-" * 40)

{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {
      "name": "my-client",
      "version": "0.1"
    }
  }
}
----------------------------------------
{
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list"
}
----------------------------------------
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "add",
    "arguments": {
      "a": 2,
      "b": 3
    }
  }
}
----------------------------------------


## And the server replies


In [2]:
# 2. MCP server replies to the client with its list of tools (id 2)
tools_response = {
    "jsonrpc": "2.0",
    "id": 2,
    "result": {
        "tools": [
            {
                "name": "add",
                "description": "Add two numbers",
                "inputSchema": {
                    "type": "object",
                    "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                    "required": ["a", "b"],
                },
            }
        ]
    },
}

# 3. MCP server runs the tool and sends the result back to the client (id 3)
call_response = {
    "jsonrpc": "2.0",
    "id": 3,
    "result": {"content": [{"type": "text", "text": "5"}]},
}

print(json.dumps(tools_response, indent=2))
print(json.dumps(call_response, indent=2))

{
  "jsonrpc": "2.0",
  "id": 2,
  "result": {
    "tools": [
      {
        "name": "add",
        "description": "Add two numbers",
        "inputSchema": {
          "type": "object",
          "properties": {
            "a": {
              "type": "integer"
            },
            "b": {
              "type": "integer"
            }
          },
          "required": [
            "a",
            "b"
          ]
        }
      }
    ]
  }
}
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "5"
      }
    ]
  }
}


## Recap

- MCP = a standard protocol between AI apps and tools
- Host → Client → Server
- Servers expose **tools, resources, prompts**
- Messages are plain JSON-RPC

**Next episode:** build a real MCP server in about 10 lines.
